# Mart Sales Prediction with CatBoost — DSN AI Bootcamp Hackathon 2026

This notebook predicts `total_sales` for the Mart Sales hackathon dataset

**Table of contents**
1. Setup & data loading
2. Exploratory data analysis
3. Data cleaning
4. Missing-value imputation & feature engineering
5. Prepare modelling data
6. Quick sanity-check fit (learning curve)
7. 5-fold cross-validation
8. Diagnostics
9. Final model & submission



## 1. Setup & data loading

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold
from sklearn import metrics

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

SEED = 42
np.random.seed(SEED)

In [ ]:
# Works whether you're on Colab, Kaggle, or running locally — no need to hardcode
# a path like "/content/train.csv" that only works in one environment.

def find_file(filename, search_dirs=(".", "./data", "/content", "/kaggle/input")):
    for d in search_dirs:
        candidate = os.path.join(d, filename)
        if os.path.exists(candidate):
            return candidate
    if os.path.exists("/kaggle/input"):
        for root, _, files in os.walk("/kaggle/input"):
            if filename in files:
                return os.path.join(root, filename)
    raise FileNotFoundError(
        f"Couldn't find '{filename}' in {search_dirs}. "
        "Update `search_dirs` above or move the file next to this notebook."
    )


train_df = pd.read_csv(find_file("train.csv"))
test_df = pd.read_csv(find_file("test.csv"))

print("train:", train_df.shape)
print("test:", test_df.shape)

## 2. Exploratory data analysis

In [ ]:
train_df.head()

In [ ]:
train_df.info()

In [ ]:
train_df.describe()

### 2.1 Missing values

`product_weight_kg` and `store_size` are the two columns with real gaps. We'll fix these properly in section 4 instead of just dumping `-999` into every column.

In [ ]:
missing = pd.DataFrame({
    "train_missing_%": train_df.isna().mean() * 100,
    "test_missing_%": test_df.reindex(columns=train_df.columns.drop("total_sales")).isna().mean() * 100,
}).round(2).sort_values("train_missing_%", ascending=False)
missing

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
missing["train_missing_%"].plot(kind="bar", ax=ax, color="#4C72B0")
ax.set_ylabel("% missing")
ax.set_title("Missing values by column (train)")
plt.tight_layout()
plt.show()

### 2.2 Target distribution

`total_sales` is right-skewed. A log transform makes it look more symmetric — worth trying as a modelling variant (see section 10), though whether it actually improves RMSE on the *original* scale needs to be checked empirically rather than assumed.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(train_df["total_sales"], kde=True, ax=axes[0], color="#4C72B0")
axes[0].set_title(f"total_sales  (skew={train_df['total_sales'].skew():.2f})")
sns.histplot(np.log1p(train_df["total_sales"]), kde=True, ax=axes[1], color="#DD8452")
axes[1].set_title(f"log1p(total_sales)  (skew={np.log1p(train_df['total_sales']).skew():.2f})")
plt.tight_layout()
plt.show()

### 2.3 Categorical values — this is where the real bugs hide



In [ ]:
cat_cols = train_df.select_dtypes(include="object").columns.drop("id")
for col in cat_cols:
    uniques = sorted(train_df[col].dropna().unique().tolist())
    print(f"{col:>22s}: {train_df[col].nunique():>3d} unique values")
    if len(uniques) <= 40:
        print("   ", uniques)

In [ ]:
# A shelf_visibility of exactly 0 doesn't make physical sense for a product that's
# on the shelf and being sold — it's the same kind of data-entry artifact that shows
# up in the classic Big Mart Sales dataset. Worth flagging before we model it.
zero_vis = (train_df["shelf_visibility"] == 0).mean()
print(f"Fraction of rows with shelf_visibility == 0: {zero_vis:.2%}")

### 2.4 Numeric correlations & categorical relationships with the target

In [ ]:
num_cols = train_df.select_dtypes(include=[np.number]).columns
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(train_df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Numeric feature correlations")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, col in zip(axes.ravel(), ["store_format", "store_location_tier", "store_size", "fat_content"]):
    sns.boxplot(data=train_df, x=col, y="total_sales", ax=ax)
    ax.set_title(f"total_sales by {col}")
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

## 3. Data cleaning



In [ ]:
def clean_fat_content(x):
    '''Standardise fat_content, collapsing known abbreviations onto one canonical spelling.'''
    if pd.isna(x):
        return x
    x = str(x).strip().lower()
    if x in ("low fat", "lf", "low_fat"):
        return "Low Fat"
    if x in ("regular", "reg"):
        return "Regular"
    return x.title()  # fallback: still normalises case/whitespace for anything unexpected


def clean_text_category(x):
    '''Generic case/whitespace normaliser for the other text categories.'''
    if pd.isna(x):
        return x
    return str(x).strip().title()


for df in (train_df, test_df):
    df["fat_content"] = df["fat_content"].apply(clean_fat_content)
    for col in ["product_category", "store_size", "store_location_tier", "store_format"]:
        df[col] = df[col].apply(clean_text_category)

print("fat_content — unique values after cleaning:", sorted(train_df["fat_content"].unique()))
print("product_category — unique values after cleaning:", train_df["product_category"].nunique())

## 4. Missing-value imputation & feature engineering



In [ ]:
train_df["is_train"] = 1
test_df["is_train"] = 0
test_df["total_sales"] = np.nan
full = pd.concat([train_df, test_df], ignore_index=True, sort=False)

# Missing-value indicator flags, captured *before* we fill anything — missingness
# itself can be predictive (e.g. maybe stores that don't report their size skew smaller).
full["product_weight_was_missing"] = full["product_weight_kg"].isna().astype(int)
full["store_size_was_missing"] = full["store_size"].isna().astype(int)

In [ ]:
# shelf_visibility == 0 -> treat as a data artifact, not a real reading.
# Replace it with the mean visibility for that product's category (computed on non-zero rows).
full["shelf_visibility_was_zero"] = (full["shelf_visibility"] == 0).astype(int)

vis_mean_by_category = (
    full.loc[full["shelf_visibility"] > 0].groupby("product_category")["shelf_visibility"].mean()
)
zero_mask = full["shelf_visibility"] == 0
full.loc[zero_mask, "shelf_visibility"] = full.loc[zero_mask, "product_category"].map(vis_mean_by_category)
full["shelf_visibility"] = full["shelf_visibility"].fillna(full["shelf_visibility"].mean())

In [ ]:
# product_weight_kg is a property of the *product*, not the store — so the best fill value
# is that product's own known weight (average across the rows where it wasn't missing).
# Falls back to the product-category mean, then the global mean, in case a product's
# weight is missing everywhere it appears.
full["product_weight_kg"] = full["product_weight_kg"].fillna(
    full.groupby("product_code")["product_weight_kg"].transform("mean")
)
full["product_weight_kg"] = full["product_weight_kg"].fillna(
    full.groupby("product_category")["product_weight_kg"].transform("mean")
)
full["product_weight_kg"] = full["product_weight_kg"].fillna(full["product_weight_kg"].mean())

In [ ]:
# store_size is a property of the *store* — same idea, but with the mode instead of
# the mean since it's categorical. Falls back to store_format's mode, then the global mode.
def safe_mode(s):
    s = s.dropna()
    return s.mode().iloc[0] if len(s) else np.nan

full["store_size"] = full["store_size"].fillna(full.groupby("store_code")["store_size"].transform(safe_mode))
full["store_size"] = full["store_size"].fillna(full.groupby("store_format")["store_size"].transform(safe_mode))
full["store_size"] = full["store_size"].fillna(safe_mode(full["store_size"]))

assert full["product_weight_kg"].isna().sum() == 0
assert full["store_size"].isna().sum() == 0
print("Remaining missing values:\n", full[["product_weight_kg", "store_size", "shelf_visibility"]].isna().sum())

### Feature engineering



In [ ]:
full["price_per_kg"] = full["product_price"] / full["product_weight_kg"].replace(0, np.nan)
full["price_per_kg"] = full["price_per_kg"].fillna(full["price_per_kg"].median())

# duplicates="drop" avoids a crash if store_age_years has few distinct values relative to
# the number of stores (quartile edges can otherwise collide)
full["store_age_quartile"] = pd.qcut(full["store_age_years"], q=4, duplicates="drop").astype(str)

cat_vis_mean = full.groupby("product_category")["shelf_visibility"].mean()
full["visibility_vs_category_mean"] = full["shelf_visibility"] - full["product_category"].map(cat_vis_mean)

full["store_num_products"] = full.groupby("store_code")["id"].transform("count")
full["store_avg_price"] = full.groupby("store_code")["product_price"].transform("mean")

train_clean = full[full["is_train"] == 1].drop(columns=["is_train"]).reset_index(drop=True)
test_clean = full[full["is_train"] == 0].drop(columns=["is_train", "total_sales"]).reset_index(drop=True)
print("train_clean:", train_clean.shape, " test_clean:", test_clean.shape)

## 5. Prepare modelling data

`cat_features` is now derived programmatically from the dataframe's dtypes instead of being a hand-typed list of column names — one less place for a typo to silently break things when a column gets renamed or a new one gets added.

In [ ]:
y = train_clean["total_sales"]
X = train_clean.drop(columns=["id", "total_sales"])
test_ids = test_clean["id"]
X_test = test_clean.drop(columns=["id"])

cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
print("Categorical features:", cat_features)
print("Numeric features:", [c for c in X.columns if c not in cat_features])

In [ ]:
%pip install -q catboost

In [ ]:
from catboost import CatBoostRegressor

## 6. Quick sanity-check fit (optional)

One fit on a single 80/20 split, mainly to *look at* the training curve (`plot=True`) and sanity-check that everything is wired up correctly before running the full cross-validation in section 7. The number here is **not** the number to trust — section 7's cross-validated RMSE is.

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=SEED)

quick_model = CatBoostRegressor(
    iterations=2000,
    depth=7,
    learning_rate=0.03,
    loss_function="RMSE",
    border_count=128,
    l2_leaf_reg=5.0,
    random_strength=2.0,
    bootstrap_type="MVS",
    bagging_temperature=0.2,
    random_seed=SEED,
    early_stopping_rounds=100,
)
quick_model.fit(X_tr, y_tr, cat_features=cat_features, eval_set=(X_val, y_val), plot=True, verbose=False)

train_rmse = metrics.root_mean_squared_error(y_tr, quick_model.predict(X_tr))
val_rmse = metrics.root_mean_squared_error(y_val, quick_model.predict(X_val))
print(f"Train RMSE: {train_rmse:.2f}")
print(f"Val   RMSE: {val_rmse:.2f}   (best iteration: {quick_model.get_best_iteration()})")

## 7. 5-fold cross-validation

A single 70/30 (or 80/20) split gives one noisy estimate of how well the model generalises, and throws away a chunk of data the final model never gets to train on. 5-fold CV instead:
- gives a mean ± std RMSE across 5 different splits, which is a much more trustworthy estimate
- uses early stopping *within* each fold, so we're not guessing the right number of iterations
- produces out-of-fold (OOF) predictions for every training row, which we use for diagnostics in section 8

This takes a few minutes to run — each of the 5 folds trains its own CatBoost model.

In [ ]:
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_pred = np.zeros(len(X))
fold_rmses = []
best_iterations = []
fold_importances = []

catboost_params = dict(
    depth=7,
    learning_rate=0.03,
    loss_function="RMSE",
    border_count=128,
    l2_leaf_reg=5.0,
    random_strength=2.0,
    bootstrap_type="MVS",
    bagging_temperature=0.2,
    random_seed=SEED,
)

for fold, (tr_idx, val_idx) in enumerate(kf.split(X), start=1):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = CatBoostRegressor(iterations=3500, early_stopping_rounds=150, verbose=False, **catboost_params)
    model.fit(X_tr, y_tr, cat_features=cat_features, eval_set=(X_val, y_val), use_best_model=True)

    fold_pred = model.predict(X_val)
    oof_pred[val_idx] = fold_pred

    fold_rmse = metrics.root_mean_squared_error(y_val, fold_pred)
    fold_rmses.append(fold_rmse)
    best_iterations.append(model.get_best_iteration())
    fold_importances.append(model.get_feature_importance())

    print(f"Fold {fold}/{N_SPLITS}  RMSE={fold_rmse:8.2f}  best_iteration={model.get_best_iteration()}")

print()
print(f"Mean fold RMSE: {np.mean(fold_rmses):.2f} +/- {np.std(fold_rmses):.2f}")
print(f"OOF RMSE:       {metrics.root_mean_squared_error(y, oof_pred):.2f}")
print(f"OOF MAE:        {metrics.mean_absolute_error(y, oof_pred):.2f}")
print(f"OOF R2:         {metrics.r2_score(y, oof_pred):.4f}")

## 8. Diagnostics

In [ ]:
mean_importance = np.mean(fold_importances, axis=0)
importance_df = pd.DataFrame({"feature": X.columns, "importance": mean_importance}).sort_values("importance")

fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(importance_df["feature"], importance_df["importance"], color="#4C72B0")
ax.set_title("CatBoost feature importance (mean over folds)")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

axes[0].scatter(y, oof_pred, alpha=0.25, s=10, color="#4C72B0")
lims = [min(y.min(), oof_pred.min()), max(y.max(), oof_pred.max())]
axes[0].plot(lims, lims, "r--", linewidth=1)
axes[0].set_xlabel("Actual total_sales")
axes[0].set_ylabel("OOF predicted total_sales")
axes[0].set_title("Predicted vs. actual (out-of-fold)")

residuals = y - oof_pred
axes[1].hist(residuals, bins=40, color="#DD8452")
axes[1].axvline(0, color="black", linewidth=1)
axes[1].set_title(f"OOF residuals  (mean={residuals.mean():.1f})")
plt.tight_layout()
plt.show()

## 9. Final model & submission

The cross-validation above tells us roughly how many iterations the model needs before it starts overfitting. For the model that actually generates the submission, we retrain on **100% of the training data** (not just 80% or 70% of it) using that iteration count — every row we have labels for should help the model that makes real predictions.

In [ ]:
final_iterations = int(np.mean(best_iterations) * 1.1) + 1
print(f"Training final model for {final_iterations} iterations "
      f"(mean best_iteration across folds was {np.mean(best_iterations):.0f})")

final_model = CatBoostRegressor(iterations=final_iterations, verbose=False, **catboost_params)
final_model.fit(X, y, cat_features=cat_features)

In [ ]:
test_predictions = final_model.predict(X_test)
test_predictions = np.clip(test_predictions, 0, None)  # sales can't be negative

sample = pd.read_csv(find_file("sample_submission.csv"))
print("sample_submission columns:", sample.columns.tolist())

submission = sample.copy()
submission[sample.columns[0]] = test_ids.values
submission[sample.columns[1]] = test_predictions
submission.to_csv("submission.csv", index=False)
submission.head()

In [ ]:
# Only relevant on Google Colab — skipped everywhere else instead of erroring out.
try:
    from google.colab import files
    files.download("submission.csv")
except ImportError:
    print("Not running on Colab — submission.csv has been saved in the working directory.")